<a href="https://colab.research.google.com/github/kardwalker/fine-tuning-a-pretrained-model/blob/main/Fine_tuning__pretrained_model(BERT).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning BERT


### Loading the Dataset


In [ ]:
!pip install datasets
!pip install transformers["sentencepiece"]
from datasets import load_dataset

dataset = load_dataset("glue","mrpc")
dataset


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [ ]:
dataset_train = dataset["train"]
print(dataset_train)

print("\n")
print("dataset_Train features")
print(dataset_train.features)


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})


dataset_Train features
{'sentence1': Value(dtype='string', id=None), 'sentence2': Value(dtype='string', id=None), 'label': ClassLabel(names=['not_equivalent', 'equivalent'], id=None), 'idx': Value(dtype='int32', id=None)}


In [ ]:
dataset_train[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

### Processing the dataset

In [ ]:
from transformers import AutoTokenizer
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenizer

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [ ]:
def tokenize_function(ex):
  return tokenizer(ex["sentence1"], ex["sentence2"], truncation=True)
#Static padding
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

Dynamic padding

In [ ]:
# function that is responsible for putting together samples inside the batch is collate function
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

samples = tokenized_datasets["train"][:8]
print(samples.items())
samples = {k:v for k,v in samples.items() if k not in ["idx","sentence1","sentence2"]}
print([len(x) for x in samples["input_ids"]])



dict_items([('sentence1', ['Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', "Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .", 'They had published an advertisement on the Internet on June 10 , offering the cargo for sale , he added .', 'Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .', 'The stock rose $ 2.11 , or about 11 percent , to close Friday at $ 21.51 on the New York Stock Exchange .', 'Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .', 'The Nasdaq had a weekly gain of 17.27 , or 1.2 percent , closing at 1,520.15 on Friday .', 'The DVD-CCA then appealed to the state Supreme Court .']), ('sentence2', ['Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .', "Yucaipa bought Dominick 's in 1995 for $ 693 mil

In [ ]:
batch = data_collator(samples)
{
    k:v.shape for k,v in batch.items()
}


{'input_ids': torch.Size([8, 67]),
 'token_type_ids': torch.Size([8, 67]),
 'attention_mask': torch.Size([8, 67]),
 'labels': torch.Size([8])}

### Fine-tuning a model with the Trainer API

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding , TrainingArguments, Trainer, AutoModelForSequenceClassification
import numpy as np
import evaluate
checkpoint = "bert-base-uncased"
raw_datasets = load_dataset("glue", "mrpc")
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
# for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


# Metrics
def compute_metrics(eval_preds):
    metric = load_metric("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments("test-trainer")
trainer = Trainer(
    model ,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics= compute_metrics,
    fp16=True,
)
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Step,Training Loss
500,0.534900


### Trianer


1. Dataset Handling and DataLoader Management
2. Device Management
3. Optimizer and Scheduler Initialization
4. Gradient Accumulation & Backpropagation
5. Logging & Metrics
6. Evaluation and Saving models
7. Mixed Precision Training


In [ ]:
import numpy as np
t = np.random.randint(0,104, 50)
i = t.reshape(2, 25)
i.shape
print("i", i)
perds = np.argmax(i, axis= -1)
perds
i[1:]
i[-1]

i [[ 56  38  34  23  69  15  61  55   4  29  24  86  23 102  20  30  44   1
   18  99  16 100  44  31  85]
 [ 87  98  14  60   5   5  45  66  58  13  36  15  11  15  75  62  43  80
   59   4  93  73  37  45  53]]


array([87, 98, 14, 60,  5,  5, 45, 66, 58, 13, 36, 15, 11, 15, 75, 62, 43,
       80, 59,  4, 93, 73, 37, 45, 53])

### without Trainer

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer , DataCollatorWithPadding , TrainingArguments, Trainer, AutoModelForSequenceClassification
raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(ex):
  return tokenizer(ex["sentence1"], ex["sentence2"] ,truncation = True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched = True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)





tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

Prepare Training


In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names


['labels', 'input_ids', 'token_type_ids', 'attention_mask']

Let's define dataloader

In [ ]:
from torch.utils.data import DataLoader
train_dataloader  = DataLoader(
    tokenized_datasets["train"], shuffle = True , batch_size = 8, collate_fn =data_collator
)
eval_dataloader =  DataLoader(
    tokenized_datasets["validation"], batch_size =8, collate_fn = data_collator

)


To check that there is no mistake in data processing

we can inspect a batch like this

In [ ]:
for batch in train_dataloader:
  break
{k: v.shape for k,v in batch.items()}
# Data preprocessing is completely finished

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 69]),
 'token_type_ids': torch.Size([8, 69]),
 'attention_mask': torch.Size([8, 69])}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels = 2)
outputs = model(**batch)
outputs.loss, outputs.logits.shape

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(tensor(0.6602, grad_fn=<NllLossBackward0>), torch.Size([8, 2]))

Now it's time to write optimizer and learning rate scheduler

In [ ]:
from transformers import AdamW
from transformers import get_scheduler

# Use the PyTorch implementation torch.optim.AdamW instead
optimizer = AdamW(model.parameters(), lr = 5e-5)
num_epochs = 3
num_training_steps = num_epochs*len(train_dataloader)
lr__scheduler = get_scheduler(
    "linear",
    optimizer = optimizer,
    num_warmup_steps = 0,
    num_training_steps = num_training_steps
)


num_training_steps

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


1377

### Training loop

we 've defined earlier works fine on a single CPU or GPU

In [ ]:
## Device Management
import torch
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device


device(type='cpu')

In [ ]:
from tqdm.auto import tqdm
progress_bar = tqdm(range(num_trainig_steps))

model.train()
for epoch in range(num_epochs):
  for batch in train_dataloader:
    batch = {k: v.to(device) for k,v in batch.items()}
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()

    optimizer.step()
    lr_scheduler.step()
    optimizer.zero_grad()
    porgress_bar.update(1)


The evaluation loop

In [ ]:
import evaluate

metric = evaluate.load("glue","mrpc")
model.eval()
for batch in eval_dataloader:
  batch = {k: v.to(device) for k,v in batch.items()}
  with torch.no_grad():
    outputs = model(**batch)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim = -1)
    metric.add_batch(predictions = predictions, references = batch["labels"])

metric.compute()

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification,
    AdamW, get_scheduler, TrainingArguments, Trainer
)
from torch.utils.data import DataLoader
import torch
from tqdm.auto import tqdm
import evaluate

# Load the dataset and tokenizer
raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples["sentence1"], examples["sentence2"], truncation=True)

# Tokenize datasets
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

# Initialize DataCollator and DataLoaders
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

# Load model and prepare it for training
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Optimizer and scheduler setup
optimizer = AdamW(model.parameters(), lr=5e-5)

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)

lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# Set device
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

# Training loop
progress_bar = tqdm(range(num_training_steps))
model.train()

for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        progress_bar.update(1)

# Evaluation
metric = evaluate.load("glue", "mrpc")
model.eval()

for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        metric.add_batch(predictions=predictions, references=batch["labels"])

# Compute final metric
result = metric.compute()
print(result)


### Enabling distributed training on multiple GPUs or TPUs

using the Accelrate library

In [ ]:
from transformers import AdamW, AutoModelForSequenceClassification, get_scheduler
from accelerate import Accelerator


accelerator : Accelerator = Acceralerator()
model = AutoModelForSequenceClassification.from_pretrained(checkpoint , num_labels= 2)
optimizer = AdamW(model.paramters(),lr = 3e-5)

#device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
#model.to(device)

train_dataloader, eval_dataloader, model , optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)

num_epoch = 3
num_training_steps = num_epochs + len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer = optimizer,
    num_warmup_steps = 0,
    num_training_steps = num_training_steps,

)
progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
  for batch in dataloader:
    outputs =  model(**batch)
    loss = outputs.loss

    accelerator.backward(loss)
    lr_scheduler.step()
    optimizer.zero_grad()
    progress_bar.update(1)
"""
 In order to benefit from the speed-up offered by Cloud TPUs, we recommend padding your
 samples to a fixed length with the `padding="max_length"` and `max_length` arguments of the tokenizer.

"""
# https://github.com/huggingface/accelerate/tree/main/examples
# accelerate repo
